# Amazon Review 2023
Amazon Reviews dataset is large-scale dataset collected in 2023 by McAuley Lab, and it includes rich features such as:
- User Reviews (ratings, text, helpfulness votes, etc.);
- Item Metadata (descriptions, price, raw image, etc.);
- Links (user-item / bought together graphs).

Related Information
- HP: https://amazon-reviews-2023.github.io/
- paper: [Bridging Language and Items for Retrieval and Recommendation](https://arxiv.org/abs/2403.03952)


In [ ]:
%load_ext autoreload

In [ ]:
%autoreload 2

import pathlib

from torch_geometric.data import HeteroData

from ml_sandbox_libs.data.amazon_reviews_dataset import (
    AmazonReviewsSeqRecDataModule,
    bipartite_graph_preprocess_dataset,
    fetch_dataset,
    fetch_metadata,
)

In [ ]:
dataset_dict = fetch_dataset(category="Video_Games", dataset_type="0core_timestamp_w_his")
df = dataset_dict["train"].to_polars()
df.head()

In [ ]:
print("Dataset Size")
print(
    f"train: {len(dataset_dict['train'])}, valid: {len(dataset_dict['valid'])}, test: {len(dataset_dict['test'])}"
)

The dataset schema is as follows: https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023#for-user-reviews

| Field            | Type     | Explanation                                                                                                                                                                               |
|------------------|----------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| user_id          | str      | ID of the reviewer                                                                                                                                                                       |
| parent_asin      | str      | Parent ID of the product. Note: Products with different colors, styles, sizes usually belong to the same parent ID. The “asin” in previous Amazon datasets is actually parent ID. **Please use parent ID to find product meta.** |
| rating           | float    | Rating of the product (from 1.0 to 5.0).                                                                                                                                                   |
| timestamp        | int      | Time of the review (unix time)                                                                                                                                                           |
| history | str     | parent_asin list which was bought by user before. The separator is ' '                                                                                                                                                               |

In [ ]:
metadata_dataset = fetch_metadata(category="Video_Games")
metadata_df = metadata_dataset.to_polars().select(
    ["parent_asin", "title", "categories", "main_category", "average_rating", "rating_number"]
)
metadata_df.head(5)

In [ ]:
print(f"Parent Asin Size: {len(metadata_df)}")

The metadata schema is as follows: https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023#for-item-metadata

| Field           | Type   | Explanation                                                                                                                                                             |
|-----------------|--------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| parent_asin     | str    | Parent ID of the product.                                                                                                                                                |
| title           | str    | Name of the product.                                                                                                                                                     |
| categories      | list   | Hierarchical categories of the product.                                                                                                                                  |


In [ ]:
datamodule.train_df.head(5)

In [ ]:
datamodule = AmazonReviewsSeqRecDataModule(
    save_dir=pathlib.Path("../data"),
    batch_size=2,
    num_workers=4,
    max_seq_len=5,
    neg_sample_size=2,
    sampling_val_test=True,
    eval_negative_sample_size=10,
    filter_no_history=False,
)

datamodule.prepare_data()
datamodule.setup(stage="fit")
train_dataloader = datamodule.train_dataloader()
batch = next(iter(train_dataloader))

In [ ]:
datamodule.train_df.head(5)

In [ ]:
batch.user_index, batch.pos_item_index, batch.neg_item_indexes, batch.item_history

## Bipartite Graph

In [ ]:
(
    all_df,
    user2index,
    item2index,
    category2index,
    item_index_2_category_index,
) = bipartite_graph_preprocess_dataset(dataset_dict=dataset_dict, metadata=metadata_dataset)

In [ ]:
all_df.head(5)

In [ ]:
import numpy as np
import polars as pl
import torch
import torch_geometric.transforms as T
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.sampler import NegativeSampling

user_index = torch.as_tensor(sorted(user2index.values()), dtype=torch.int64)

In [ ]:
item_df = pl.from_dict({"item_index": item2index.values()})
item2category_df = pl.from_dict(
    {
        "item_index": item_index_2_category_index.keys(),
        "category": item_index_2_category_index.values(),
    }
)
item_df = item_df.join(item2category_df, on="item_index", validate="m:1").sort("item_index")
item_index = item_df["item_index"].to_torch()
category_index = item_df["category"].to_torch()

In [ ]:
all_df.filter(pl.col("split") == "train")

In [ ]:
def create_bipartite_graph(
    split: str,
    all_df: pl.DataFrame,
    user2index: dict[str, int],
    item2index: dict[str, int],
    item_index_2_category_index: dict[int, int],
):
    user_index = torch.as_tensor(sorted(user2index.values()), dtype=torch.int64)
    item_df = pl.from_dict({"item_index": item2index.values()})
    item2category_df = pl.from_dict(
        {
            "item_index": item_index_2_category_index.keys(),
            "category": item_index_2_category_index.values(),
        }
    )
    item_df = item_df.join(item2category_df, on="item_index", validate="m:1").sort("item_index")
    item_index = item_df["item_index"].to_torch()
    category_index = item_df["category"].to_torch()

    match split:
        case "train":
            df = all_df.filter(pl.col("split") == "train")
        case "valid":
            df = all_df.filter(pl.col("split").is_in(["train", "valid"]))
        case "test":
            df = all_df.filter(pl.col("split").is_in(["train", "valid", "test"]))
        case _:
            raise ValueError(f"Invalid split: {split}")

    edge_index = torch.as_tensor(
        np.ascontiguousarray(df["user_index", "item_index"].to_numpy().T), dtype=torch.long
    )
    edge_label_index = torch.as_tensor(
        np.ascontiguousarray(
            df.filter(pl.col("split") == split)["user_index", "item_index"].to_numpy().T
        ),
        dtype=torch.long,
    )
    data = HeteroData(
        {
            "user": {"x": user_index.unsqueeze(-1), "user_index": user_index},
            "item": {
                "x": item_index.unsqueeze(-1),
                "item_index": item_index,
                "category_index": category_index,
            },
            ("user", "rates", "item"): {
                "edge_index": edge_index,
                "edge_label_index": edge_label_index,
            },
        }
    )
    return data

In [ ]:
train_data = create_bipartite_graph(
    split="valid",
    all_df=all_df,
    user2index=user2index,
    item2index=item2index,
    item_index_2_category_index=item_index_2_category_index,
)
val_data = create_bipartite_graph(
    split="valid",
    all_df=all_df,
    user2index=user2index,
    item2index=item2index,
    item_index_2_category_index=item_index_2_category_index,
)

transform = T.Compose([T.RemoveIsolatedNodes(), T.RemoveSelfLoops()])
train_data = transform(train_data)
val_data = transform(val_data)

In [ ]:
train_data

In [ ]:
neg_sampling = NegativeSampling(mode="triplet", amount=3)

loader = LinkNeighborLoader(
    data=train_data,
    num_neighbors=[10, 5],
    batch_size=2,
    edge_label_index=(
        ("user", "rates", "item"),
        train_data["user", "rates", "item"].edge_label_index,
    ),
    edge_label=None,
    neg_sampling=neg_sampling,
    shuffle=True,
)

In [ ]:
train_data["user", "rates", "item"].edge_index

In [ ]:
train_data.keys()

In [ ]:
val_data["user"].x.size()

In [ ]:
train_df = all_df.filter(pl.col("split") == "train")
user_item_edge_index = train_df["user_index", "item_index"].to_torch(dtype=pl.Int64)

data = HeteroData(
    {
        "user": {"x": user_index, "user_index": user_index},
        "item": {"x": user_index, "item_index": item_index, "category_index": category_index},
        ("user", "rates", "item"): {"edge_index": user_item_edge_index},
    }
)
transform = T.Compose([T.RemoveIsolatedNodes(), T.RemoveSelfLoops()])
data = transform(data)

In [ ]:
data = HeteroData(
    {
        "user": {"user_index": user_index},
        "item": {"item_index": item_index, "category_index": category_index},
        ("user", "rates", "item"): {"edge_index": user_item_edge_index},
    }
)

In [ ]:
data